In [1]:
import os, glob

PROJECT_FOLDER = "project_data"
os.makedirs(PROJECT_FOLDER, exist_ok=True)

print("Folder ready:", PROJECT_FOLDER)

Folder ready: project_data


In [13]:
project_queries = {
    "Q1": {
        "query": "Under the Co-Branding Agreement, what is the exact formula used to calculate the advertising fees i-Escrow must pay 2TheMart?",
        "relevant_evidence": ["Section 5.1 'ADVERTISING FEES'", "Transaction Inquiry"],
        "correct_answer": "The advertising fee is based on a per Transaction Inquiry amount, calculated by multiplying 0.025% by the average Transaction amount."
    },
    "Q2": {
        "query": "What are the specific conditions and timeline under which Adams Golf can terminate the endorsement agreement with Tom Watson for cause?",
        "relevant_evidence": ["Section 23 'TERMINATION FOR CAUSE'"],
        "correct_answer": "Adams Golf can terminate for cause if Tom Watson commits a material breach, provided 15 days written notice is given and the breach is not rectified within that 15-day period."
    },
    "Q3_ambiguous": {
        "query": "According to the Adams Golf Endorsement Agreement, what specific brand of putter is Tom Watson contractually required to use during the US Open?",
        "relevant_evidence": ["Section 5 'EXCLUSIVE USE OF PRODUCT'", "parenthetical exception"],
        "correct_answer": "The premise is false. Tom Watson is not required to use an Adams Golf putter; he may play another manufacturer's putter during the US Open as long as he does not endorse it."
    }
}

In [ ]:
%pip install -q sentence-transformers faiss-cpu chromadb datasets transformers scikit-learn rank-bm25

import os, glob, re
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Set

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer
import faiss

from transformers import pipeline

print(" Setup complete. If you see dependency warnings, ignore unless imports fail.")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


/Users/ahirmansi1301gmail.com/LexGaurd/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Setup complete. If you see dependency warnings, ignore unless imports fail.


In [4]:
# Benchmark Loader (classroom-safe fallback; avoids script-based datasets)
def load_benchmark(n: int = 120) -> List[str]:
    # 1) Try a script-free SciFact source
    try:
        print("Trying allenai/scifact...")
        ds = load_dataset("allenai/scifact", split=f"train[:{n}]")
        sample = ds[0]
        if "claim" in sample:
            return [x["claim"] for x in ds]
        if "text" in sample:
            return [x["text"] for x in ds]
        raise RuntimeError("Unknown SciFact schema.")
    except Exception as e:
        print("⚠️ allenai/scifact failed:", str(e))

    # 2) Try multi_news
    try:
        print("Trying multi_news...")
        ds = load_dataset("multi_news", split=f"train[:{n}]")
        return [x["document"] for x in ds]
    except Exception as e:
        print("⚠️ multi_news failed:", str(e))

    # 3) Fallback: ag_news (very stable)
    print("Using ag_news fallback...")
    ds = load_dataset("ag_news", split=f"train[:{n}]")
    return [x["text"] for x in ds]

# Load benchmark docs
benchmark_docs = load_benchmark(n=120)
print(f"Loaded benchmark docs: {len(benchmark_docs)}")

# Load project-aligned docs from ./project_data/*.txt
PROJECT_FOLDER = "project_data"
project_files = sorted(glob.glob(os.path.join(PROJECT_FOLDER, "*.txt")))
project_docs = []
for fp in project_files:
    with open(fp, "r", encoding="utf-8", errors="ignore") as f:
        project_docs.append(f.read())

print(f"Loaded project docs: {len(project_docs)}")
if len(project_docs) == 0:
    print("⚠️ Add 3–20 .txt files under ./project_data/ to earn full credit.")

Trying allenai/scifact...
⚠️ allenai/scifact failed: Dataset scripts are no longer supported, but found scifact.py
Trying multi_news...
⚠️ multi_news failed: Dataset scripts are no longer supported, but found multi_news.py
Using ag_news fallback...


Generating test split: 100%|██████████| 7600/7600 [00:00<00:00, 2279350.05 examples/s]

Loaded benchmark docs: 120
Loaded project docs: 5


In [5]:
# --- Chunking functions ---
def fixed_chunks(text: str, size: int = 1200, overlap: int = 200) -> List[str]:
    """Character-based fixed window chunking (fast and reliable in class)."""
    text = text.strip()
    if not text:
        return []
    chunks = []
    step = max(1, size - overlap)
    for i in range(0, len(text), step):
        c = text[i:i+size].strip()
        if len(c) > 50:
            chunks.append(c)
    return chunks

def semantic_chunks(text: str) -> List[str]:
    """Paragraph-based semantic chunking; merges short segments to keep context."""
    paras = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    merged, buf = [], ""
    for p in paras:
        if len(buf) < 400:
            buf = (buf + "\n\n" + p).strip()
        else:
            merged.append(buf); buf = p
    if buf:
        merged.append(buf)
    return [m for m in merged if len(m) > 80]

def build_corpus(docs: List[str], mode: str) -> List[str]:
    all_chunks = []
    for d in docs:
        if mode == "fixed":
            all_chunks.extend(fixed_chunks(d))
        elif mode == "semantic":
            all_chunks.extend(semantic_chunks(d))
        else:
            raise ValueError("mode must be 'fixed' or 'semantic'")
    return all_chunks

# Build both corpora and choose one to use in retrieval
all_docs = benchmark_docs + project_docs
fixed_corpus = build_corpus(all_docs, mode="fixed")
semantic_corpus = build_corpus(all_docs, mode="semantic")

print("Fixed corpus chunks:", len(fixed_corpus))
print("Semantic corpus chunks:", len(semantic_corpus))

# Choose the corpus for the lab (recommend semantic for better context)
CORPUS = semantic_corpus
print("Using CORPUS =", "semantic" if CORPUS is semantic_corpus else "fixed")

Fixed corpus chunks: 301
Semantic corpus chunks: 310
Using CORPUS = semantic


In [6]:
# --- Keyword Retrieval (TF-IDF + BM25) ---
def tokenize(s: str) -> List[str]:
    return re.findall(r"[A-Za-z0-9]+", s.lower())

tfidf = TfidfVectorizer(stop_words="english", max_features=50000)
tfidf_matrix = tfidf.fit_transform(CORPUS)

def keyword_tfidf(query: str, k: int = 10) -> List[Tuple[int, float]]:
    q_vec = tfidf.transform([query])
    scores = (tfidf_matrix @ q_vec.T).toarray().squeeze()
    top = np.argsort(scores)[-k:][::-1]
    return [(int(i), float(scores[i])) for i in top]

bm25 = BM25Okapi([tokenize(x) for x in CORPUS])

def keyword_bm25(query: str, k: int = 10) -> List[Tuple[int, float]]:
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(scores)[-k:][::-1]
    return [(int(i), float(scores[i])) for i in top]

# --- Vector Retrieval (SentenceTransformer + FAISS) ---
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

embeddings = embedder.encode(CORPUS, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)  # cosine via normalized vectors + inner product
faiss_index.add(embeddings)

def vector_search(query: str, k: int = 10) -> List[Tuple[int, float]]:
    q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, idx = faiss_index.search(q, k)
    return [(int(i), float(s)) for i, s in zip(idx[0], scores[0])]

print("✅ Retrieval engines ready: TF-IDF, BM25, Vector(FAISS)")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1101.76it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 10/10 [00:04<00:00,  2.31it/s]

✅ Retrieval engines ready: TF-IDF, BM25, Vector(FAISS)


In [7]:
def normalize_scores(pairs: List[Tuple[int, float]]) -> Dict[int, float]:
    if not pairs:
        return {}
    vals = np.array([s for _, s in pairs], dtype=float)
    vmin, vmax = vals.min(), vals.max()
    if vmax - vmin < 1e-9:
        return {i: 1.0 for i, _ in pairs}
    return {i: (s - vmin) / (vmax - vmin) for i, s in pairs}

def hybrid_search(query: str, k_keyword: int = 10, k_vector: int = 10, alpha: float = 0.5,
                  top_k: int = 10, keyword_mode: str = "bm25") -> List[Tuple[int, float]]:
    kw = keyword_bm25(query, k=k_keyword) if keyword_mode == "bm25" else keyword_tfidf(query, k=k_keyword)
    vec = vector_search(query, k=k_vector)

    kw_n = normalize_scores(kw)
    vec_n = normalize_scores(vec)

    all_ids = set(kw_n) | set(vec_n)
    combined = []
    for i in all_ids:
        score = alpha * kw_n.get(i, 0.0) + (1 - alpha) * vec_n.get(i, 0.0)
        combined.append((i, float(score)))

    combined.sort(key=lambda x: x[1], reverse=True)
    return combined[:top_k]

print("✅ Hybrid retrieval ready. You'll sweep alpha ∈ {0.2, 0.5, 0.8}.")

✅ Hybrid retrieval ready. You'll sweep alpha ∈ {0.2, 0.5, 0.8}.


In [8]:
USE_CROSS_ENCODER = True
reranker = None

if USE_CROSS_ENCODER:
    try:
        from sentence_transformers import CrossEncoder
        reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        print("✅ Cross-encoder reranker loaded.")
    except Exception as e:
        print("⚠️ Cross-encoder not available. Falling back to no reranking.")
        print("Error:", e)
        reranker = None

def rerank(query: str, candidates: List[Tuple[int, float]], top_k: int = 5) -> List[Tuple[int, float]]:
    ids = [i for i, _ in candidates]
    if reranker is None:
        return candidates[:top_k]
    pairs = [(query, CORPUS[i]) for i in ids]
    scores = reranker.predict(pairs)
    scored = list(zip(ids, scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [(int(i), float(s)) for i, s in scored[:top_k]]

print("✅ Reranking function ready.")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1931.75it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Cross-encoder reranker loaded.
✅ Reranking function ready.


In [17]:
import os
import time
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY is missing! Check your .env file.")

client = Groq(api_key=api_key)
MODEL = "llama-3.3-70b-versatile"

def _groq_call(messages: list, max_tokens: int) -> str:
    """Single Groq call with 429 retry logic."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        if "429" in str(e):
            print("Rate limit hit — waiting 60s before retry...")
            time.sleep(60)
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content
        raise

def prompt_only_answer(query: str, max_new_tokens: int = 200) -> str:
    return _groq_call([{"role": "user", "content": query}], max_new_tokens)

def rag_answer(query: str, chunk_ids: List[int], max_new_tokens: int = 220) -> str:
    evidence = "\n\n".join([f"[Chunk {j+1}] {CORPUS[i]}" for j, i in enumerate(chunk_ids)])
    prompt = (
        f"Answer the question using ONLY the evidence below.\n\n"
        f"Evidence:\n{evidence}\n\n"
        f"Question:\n{query}\n\n"
        f"Rules:\n- If evidence is insufficient, say: Not enough evidence.\n"
        f"- Cite evidence with [Chunk 1], [Chunk 2], etc.\n"
    )
    return _groq_call([{"role": "user", "content": prompt}], max_new_tokens)

def show_top(pairs: List[Tuple[int, float]], title: str, k: int = 5):
    print(f"\n=== {title} (Top {k}) ===")
    for r, (i, s) in enumerate(pairs[:k], 1):
        snip = CORPUS[i].replace("\n", " ")
        snip = snip[:220] + ("..." if len(snip) > 220 else "")
        print(f"{r:>2}. id={i:<6} score={s:>8.4f} | {snip}")

queries = [
    "Q1: " + project_queries["Q1"]["query"],
    "Q2: " + project_queries["Q2"]["query"],
    "Q3 (ambiguous): " + project_queries["Q3_ambiguous"]["query"],
]

alphas = [0.2, 0.5, 0.8]
results_summary = []

for q in queries:
    print("\n" + "="*90)
    print(q)

    kw = keyword_bm25(q, k=10)
    vec = vector_search(q, k=10)
    show_top(kw, "BM25 Keyword")
    show_top(vec, "Vector (FAISS)")

    hybrids = []
    for a in alphas:
        hyb = hybrid_search(q, alpha=a, top_k=10, keyword_mode="bm25")
        hybrids.append((a, hyb))
        show_top(hyb, f"Hybrid (alpha={a})")

    best_a, _ = max(hybrids, key=lambda t: np.mean([s for _, s in t[1]]) if t[1] else -1)
    print(f"\nSelected hybrid alpha={best_a}")

    candidate_pool = hybrid_search(q, alpha=best_a, top_k=20, keyword_mode="bm25")
    reranked = rerank(q, candidate_pool, top_k=5)
    show_top(reranked, "Re-ranked")

    top3_ids = [i for i, _ in reranked[:3]]
    print("\nTop-3 evidence chunk IDs:", top3_ids)

    po = prompt_only_answer(q)
    time.sleep(15)          # respect 6000 TPM limit between calls
    ra = rag_answer(q, top3_ids)
    time.sleep(15)

    print("\n--- Prompt-only answer ---\n", po)
    print("\n--- RAG-grounded answer ---\n", ra)

    results_summary.append({
        "query": q,
        "best_alpha": best_a,
        "top3_chunk_ids": top3_ids,
        "prompt_only": po,
        "rag": ra,
    })

results_summary[:1]


Q1: Under the Co-Branding Agreement, what is the exact formula used to calculate the advertising fees i-Escrow must pay 2TheMart?

=== BM25 Keyword (Top 5) ===
 1. id=133    score= 37.8109 | (c) By adding links to Co-Branded Site on the seller listing pages of 2TheMart auctions.  (d) By displaying a text or graphic link to a page containing information about Services on all auction item pages and bidding pag...
 2. id=120    score= 34.9754 | CO-BRANDING AND ADVERTISING AGREEMENT  THIS CO-BRANDING AND ADVERTISING AGREEMENT (the "Agreement") is made as of June 21, 1999 (the "Effective Date") by and between I-ESCROW, INC., with its principal place of business a...
 3. id=130    score= 28.9959 | 2.4 RESTRICTIONS ON COMMUNICATIONS.  i-Escrow may place banner advertising on the Co-Branded Site upon prior written approval of 2TheMart, which shall be at the discretion of 2TheMart.  All advertising revenue arising fr...
 4. id=122    score= 26.1229 | (b) "CO-BRANDED SITE" means the web-site ac

[{'query': 'Q1: Under the Co-Branding Agreement, what is the exact formula used to calculate the advertising fees i-Escrow must pay 2TheMart?',
  'best_alpha': 0.8,
  'top3_chunk_ids': [133, 134, 120],
  'prompt_only': "Based on the provided information, I couldn't find the exact formula used to calculate the advertising fees that i-Escrow must pay 2TheMart under the Co-Branding Agreement. If you could provide more context or clarify the agreement, I would be happy to try and assist you further.",
  'rag': 'The formula used to calculate the advertising fees is: 0.025% multiplied by the amount of the average Transaction from all Customers in the preceding quarter. [Chunk 1]'}]

In [18]:
def precision_at_k(retrieved: List[int], relevant: Set[int], k: int = 5) -> float:
    top = retrieved[:k]
    if not top:
        return 0.0
    return sum(1 for i in top if i in relevant) / len(top)

def recall_at_k(retrieved: List[int], relevant: Set[int], k: int = 10) -> float:
    if not relevant:
        return 0.0
    return len(set(retrieved[:k]) & relevant) / len(relevant)

# ✅ REQUIRED: Label a small set of relevant chunk IDs for each query (after inspecting retrieval results).
relevance_labels = {
    queries[0]: {133, 134}, # Q1: Advertising fees formula
    queries[1]: {192},      # Q2: Termination for cause timeline
    queries[2]: {170}       # Q3: The putter exception clause
}
relevance_labels

{'Q1: Under the Co-Branding Agreement, what is the exact formula used to calculate the advertising fees i-Escrow must pay 2TheMart?': {133,
  134},
 'Q2: What are the specific conditions and timeline under which Adams Golf can terminate the endorsement agreement with Tom Watson for cause?': {192},
 'Q3 (ambiguous): According to the Adams Golf Endorsement Agreement, what specific brand of putter is Tom Watson contractually required to use during the US Open?': {170}}

In [19]:
def evaluate_query(q: str, relevant: Set[int], alpha: float):
    kw_ids = [i for i, _ in keyword_bm25(q, k=10)]
    vec_ids = [i for i, _ in vector_search(q, k=10)]
    hyb_ids = [i for i, _ in hybrid_search(q, alpha=alpha, top_k=10, keyword_mode="bm25")]
    return {
        "P@5_keyword": precision_at_k(kw_ids, relevant, k=5),
        "R@10_keyword": recall_at_k(kw_ids, relevant, k=10),
        "P@5_vector": precision_at_k(vec_ids, relevant, k=5),
        "R@10_vector": recall_at_k(vec_ids, relevant, k=10),
        "P@5_hybrid": precision_at_k(hyb_ids, relevant, k=5),
        "R@10_hybrid": recall_at_k(hyb_ids, relevant, k=10),
    }

metrics_rows = []
for row in results_summary:
    q = row["query"]
    alpha = row["best_alpha"]
    rel = relevance_labels.get(q, set())
    m = evaluate_query(q, rel, alpha)
    m.update({"query": q, "alpha_used": alpha, "num_relevant_labeled": len(rel)})
    metrics_rows.append(m)

metrics_df = pd.DataFrame(metrics_rows)
metrics_df

,P@5_keyword,R@10_keyword,P@5_vector,R@10_vector,P@5_hybrid,R@10_hybrid,query,alpha_used,num_relevant_labeled
0,0.2,0.5,0.4,1.0,0.2,1.0,"Q1: Under the Co-Branding Agreement, what is t...",0.8,2
1,0.2,1.0,0.0,1.0,0.0,0.0,Q2: What are the specific conditions and timel...,0.2,1
2,0.2,1.0,0.2,1.0,0.2,1.0,Q3 (ambiguous): According to the Adams Golf En...,0.8,1
